# 4D Phantom: Test Individual `invert` Solvers

This notebook runs one `invert` solver at a time on the MNE 4D BTi phantom dataset and reports localization error against known dipole locations.

You can switch the solver by changing `SOLVER_NAME` in the config cell.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os.path as op

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from mne.datasets import phantom_4dbti

from invert import Solver

## Config


In [ ]:
SOLVER_NAME = "eloreta"  # e.g. "mne", "wmne", "loreta", "esmv", ...
SOLVER_KWARGS = {}

RUN_IDS = [1, 2, 3, 4]
RAW_TEMPLATE = "{run_id}/e,rfhp1.0Hz"
BAD_CHANNELS = ["A173", "A213", "A232"]
EVENT_CHANNEL = "TRIGGER"
EVENT_ID = 8192
EVENT_MASK = 4350

TMIN, TMAX = -0.2, 0.4
PEAK_TIME = 0.07  # seconds

HEAD_RADIUS = 0.080  # meters
SOURCE_RADIUS = 0.072  # meters
SOURCE_GRID_MM = 8.0
MINDIST_MM = 5.0

## Ground truth dipole locations

From the MNE phantom tutorial (converted to head coordinates, meters).


In [ ]:
actual_pos = 0.01 * np.array(
    [
        [0.16, 1.61, 5.13],
        [0.17, 1.35, 4.15],
        [0.16, 1.05, 3.19],
        [0.13, 0.80, 2.26],
    ]
)
actual_pos = actual_pos @ np.array([[0, 1, 0], [-1, 0, 0], [0, 0, 1]])
actual_pos

## Helper functions

### Source space construction

We build a spherical volume source space in two steps:

1. **Regular grid**: `mne.setup_volume_source_space` creates a coarse 8 mm grid inside a 72 mm sphere (`mindist=5 mm` from the origin). This is a standard volumetric source space.
2. **Inject true dipole locations**: We append the 4 known phantom dipole coordinates to the grid and deduplicate (1 µm tolerance). This ensures the true positions are available as candidate sources, removing source-space discretization as a confound when evaluating solver accuracy. Normals are set to radial (pointing outward from the sphere center).

The combined point set is passed back to `setup_volume_source_space` via the `pos={"rr": rr, "nn": nn}` dict interface with `mindist=0` so no points are culled.

In [ ]:
def load_phantom_run(run_id, data_path):
    raw_fname = op.join(data_path, RAW_TEMPLATE.format(run_id=run_id))
    raw = mne.io.read_raw_bti(
        raw_fname, rename_channels=False, preload=True, verbose=False
    )
    raw.info["bads"] = BAD_CHANNELS

    events = mne.find_events(
        raw,
        stim_channel=EVENT_CHANNEL,
        mask=EVENT_MASK,
        mask_type="not_and",
        verbose=False,
    )
    epochs = mne.Epochs(
        raw,
        events=events,
        event_id=EVENT_ID,
        tmin=TMIN,
        tmax=TMAX,
        preload=True,
        baseline=(None, 0.0),
        verbose=False,
    )
    evoked = epochs.average()
    evoked.drop_channels(evoked.info["bads"])
    cov = mne.compute_covariance(epochs, tmax=0.0, verbose=False)
    return raw, epochs, evoked, cov


def make_sphere_forward(info):
    sphere = mne.make_sphere_model(
        r0=(0.0, 0.0, 0.0), head_radius=HEAD_RADIUS, verbose=False
    )

    # Step 1: regular volume grid
    src_grid = mne.setup_volume_source_space(
        subject=None,
        pos=SOURCE_GRID_MM,
        sphere=(0.0, 0.0, 0.0, SOURCE_RADIUS),
        mindist=MINDIST_MM,
        exclude=0.0,
        verbose=False,
    )

    # Step 2: append true dipole positions and deduplicate
    rr_used = src_grid[0]["rr"][src_grid[0]["vertno"]]
    rr = np.vstack([rr_used, actual_pos])
    rr = np.unique(np.round(rr, 6), axis=0)  # ~1 µm tolerance

    # Radial normals (outward from sphere center)
    norms = np.linalg.norm(rr, axis=1, keepdims=True)
    nn = np.zeros_like(rr)
    mask = norms.squeeze() > 0
    nn[mask] = rr[mask] / norms[mask]
    nn[~mask] = np.array([1.0, 0.0, 0.0])

    # Step 3: build final source space from the combined point set
    src = mne.setup_volume_source_space(
        subject=None,
        pos={"rr": rr, "nn": nn},
        sphere=(0.0, 0.0, 0.0, SOURCE_RADIUS),
        mindist=0.0,  # keep all points, including injected ones
        exclude=0.0,
        verbose=False,
    )
    fwd = mne.make_forward_solution(
        info,
        trans=None,
        src=src,
        bem=sphere,
        meg=True,
        eeg=False,
        verbose=False,
    )
    return fwd, sphere


def source_positions_from_forward(fwd):
    return np.concatenate([s["rr"][s["vertno"]] for s in fwd["src"]], axis=0)


def estimate_peak_position_from_matrix(source_mat, fwd):
    # collapse time by max abs value, then pick strongest source
    source_power = np.max(np.abs(source_mat), axis=1)
    peak_idx = int(np.argmax(source_power))
    pos = source_positions_from_forward(fwd)[peak_idx]
    return pos, peak_idx, source_power[peak_idx]


def run_invert_solver(evoked, fwd, cov, solver_name=SOLVER_NAME, solver_kwargs=None):
    if solver_kwargs is None:
        solver_kwargs = {}

    solver = Solver(solver_name, plot_reg=True, **solver_kwargs)
    # solver.make_inverse_operator(fwd, evoked)
    # A/B: uncomment above and comment below to skip noise covariance
    solver.make_inverse_operator(fwd, evoked, noise_cov=cov)
    stc = solver.apply_inverse_operator(evoked.copy().crop(PEAK_TIME, PEAK_TIME))
    return solver, stc


def run_mne_dipole_fit(evoked, cov, sphere):
    dip = mne.fit_dipole(
        evoked.copy().crop(PEAK_TIME, PEAK_TIME), cov, sphere, verbose=False
    )[0]
    return dip.pos[0], dip.ori[0], dip.gof[0]

## Load all runs


In [ ]:
data_path = phantom_4dbti.data_path()

runs = {}
for run_id in RUN_IDS:
    raw, epochs, evoked, cov = load_phantom_run(run_id, data_path)
    fwd, sphere = make_sphere_forward(evoked.info)
    runs[run_id] = {
        "raw": raw,
        "epochs": epochs,
        "evoked": evoked,
        "cov": cov,
        "fwd": fwd,
        "sphere": sphere,
    }

print(f"Loaded {len(runs)} phantom runs")

## Quick look at one evoked response


In [ ]:
runs[1]["evoked"].plot(time_unit="s")
plt.show()

## Evaluate the selected `invert` solver on all 4 phantom positions


In [ ]:
results = []
solver_positions = []
mne_positions = []

for run_id in RUN_IDS:
    evoked = runs[run_id]["evoked"]
    cov = runs[run_id]["cov"]
    fwd = runs[run_id]["fwd"]
    sphere = runs[run_id]["sphere"]

    # invert solver
    solver, stc = run_invert_solver(evoked, fwd, cov, SOLVER_NAME, SOLVER_KWARGS)
    pos_hat, peak_idx, peak_amp = estimate_peak_position_from_matrix(stc.data, fwd)

    # MNE dipole fit baseline from official tutorial
    dip_pos, dip_ori, dip_gof = run_mne_dipole_fit(evoked, cov, sphere)

    gt = actual_pos[run_id - 1]
    err_solver_mm = 1e3 * np.linalg.norm(pos_hat - gt)
    err_mne_mm = 1e3 * np.linalg.norm(dip_pos - gt)

    solver_positions.append(pos_hat)
    mne_positions.append(dip_pos)

    results.append(
        {
            "run": run_id,
            "solver": SOLVER_NAME,
            "solver_err_mm": err_solver_mm,
            "solver_peak_idx": peak_idx,
            "solver_peak_amp": float(peak_amp),
            "mne_dipfit_err_mm": err_mne_mm,
            "mne_dipfit_gof": float(dip_gof),
        }
    )

results_df = pd.DataFrame(results)
results_df

## Summary errors


In [ ]:
summary = {
    "solver": SOLVER_NAME,
    "solver_mean_mm": results_df["solver_err_mm"].mean(),
    "solver_std_mm": results_df["solver_err_mm"].std(ddof=0),
    "mne_dipfit_mean_mm": results_df["mne_dipfit_err_mm"].mean(),
    "mne_dipfit_std_mm": results_df["mne_dipfit_err_mm"].std(ddof=0),
}
summary

## 3D visualization: ground truth vs estimates

Uses `mne.viz.plot_dipole_locations` to render dipole positions on the sphere model.

- **Red**: ground truth phantom dipole positions
- **Yellow**: `invert` solver peak locations

In [ ]:
solver_positions = np.asarray(solver_positions)

# Build MNE Dipole objects for 3D visualization
dummy_times = np.arange(len(RUN_IDS), dtype=float)
dummy_amp = np.ones(len(RUN_IDS))
dummy_gof = np.ones(len(RUN_IDS))

# Radial orientations (pointing outward from origin)
ori = actual_pos / np.linalg.norm(actual_pos, axis=1, keepdims=True)

dip_est = mne.Dipole(dummy_times, solver_positions, dummy_amp, ori, dummy_gof)
dip_true = mne.Dipole(dummy_times, actual_pos, dummy_amp, ori, dummy_gof)

sphere = runs[1]["sphere"]
evoked = runs[1]["evoked"]

fig = mne.viz.plot_alignment(evoked.info, bem=sphere, surfaces=[])
fig = mne.viz.plot_dipole_locations(
    dipoles=dip_true, mode="sphere", color=(1.0, 0.0, 0.0), scale=0.01, fig=fig
)
fig = mne.viz.plot_dipole_locations(
    dipoles=dip_est, mode="sphere", color=(1.0, 1.0, 0.0), scale=0.01, fig=fig
)